In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
# Point these at your own data / checkpoints, or set the env vars before launching
# Jupyter. Expected layout under DATA_ROOT:
#   angle_cls/{train,test}/{normal,abnormal}/*.jpg
import os

DATA_ROOT = os.environ.get("FLATFISH_DATA_ROOT", "./data")
CKPT_DIR = os.environ.get("FLATFISH_CKPT_DIR", "./ckpt")
os.makedirs(CKPT_DIR, exist_ok=True)
# ───────────────────────────────────────────────────────────────────────────────


In [ ]:
import torch
import torch.nn as nn

from dataset import *

from tqdm import tqdm
from torch.utils.data import DataLoader

In [ ]:
device = "cuda:0"

angle_cls_model = torch.load(f"{CKPT_DIR}/angle_cls.pth").to(device)

# angle_cls_model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18')
# angle_cls_model.fc = nn.Linear(512, 2)
# angle_cls_model = angle_cls_model.to(device)

In [ ]:
root_path = f"{DATA_ROOT}/angle_cls/"

dataset = Angle_cls_dataset(root_path, 'train')

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [ ]:
optimizer = torch.optim.AdamW(
    angle_cls_model.parameters(),
    lr=0.007
)
criterion = nn.CrossEntropyLoss()

In [ ]:
angle_cls_model.train()

num_epochs = 30

for epoch in range(num_epochs):
    total_loss = 0
    for _, item in enumerate(tqdm(dataloader)):
        x, y = item
        x = x.to(device)
        y = y.squeeze(1).long().to(device)

        outputs = angle_cls_model(x)
        
        loss = criterion(outputs, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)

    print(f"[Epoch {epoch+1}/{num_epochs}] Loss: {avg_loss:.4f}")

    torch.save(angle_cls_model.state_dict(), f"trans_ckpt2/{epoch+1}.pth")

In [ ]:
# angle_cls_model = torch.load(f"{CKPT_DIR}/angle_cls.pth")


angle_cls_model = torch.hub.load('pytorch/vision:v0.10.0', 'resnet18')
angle_cls_model.fc = nn.Linear(512, 2)
angle_cls_model = angle_cls_model.to(device)

ckpt = torch.load(f"{CKPT_DIR}/16.pth")
angle_cls_model.load_state_dict(ckpt)
angle_cls_model.to(device)

In [ ]:
p = f"{DATA_ROOT}/20251028/"
filenames = os.listdir(p)

angle_cls_model.eval()
for filename in filenames:
    with torch.no_grad():
        transform=transforms.Compose([
                            transforms.Resize((256,256)),
                            transforms.ToTensor(), 
                        ])

        img = Image.open(os.path.join(p, filename))

        img = transform(img).to(device)
        output = angle_cls_model(img.unsqueeze(0))
        _, pred = output.max(dim=1)

    print(pred)

In [ ]:
root_path = f"{DATA_ROOT}/angle_cls/"

# angle_cls_model = torch.load(f"{CKPT_DIR}/angle_cls.pth").to(device)

test_dataset = Angle_cls_dataset(root_path, 'test')

test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
preds_all = []
labels_all = []

angle_cls_model.eval()

t_l = 0
with torch.no_grad():
    for _, item in enumerate(tqdm(test_dataloader)):
        x, y = item
        x = x.to(device)
        y = y.squeeze(1).long().to(device)

        outputs = angle_cls_model(x)

        loss = criterion(outputs, y)

        preds_all.append(torch.argmax(outputs, dim=1))
        labels_all.append(y)

        t_l += loss.item()
    t_l = t_l / len(test_dataloader)
preds_all = torch.cat(preds_all).cpu()
labels_all = torch.cat(labels_all).cpu()

print(t_l)

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

print(classification_report(labels_all, preds_all))
print(confusion_matrix(labels_all, preds_all))

In [ ]:
preds_all

In [ ]:
labels_all